# AI-Lake + DuckDB

Two ways to use AI-Lake from DuckDB: **standard Parquet** (§1-9, no plugin — the HNSW
footer bytes are invisible to DuckDB, which reads tabular columns normally) and the
**`duckdb-ailake` extension** (§10-14 — native vector search, hybrid FTS, and writes as
SQL functions, bridged to the same C-ABI the Spark/Trino/Flink plugins use).

**What this notebook covers:**
1. Direct Parquet scan (glob)
2. Filtered queries + aggregations
3. `embedding` column as `BLOB`
4. Iceberg extension scan (optional)
5. Performance: row count, column stats
6. `ailake_search`/`ailake_scan`/`ailake_search_text`/`ailake_search_multimodal`/`ailake_write_batch` — native SQL (§10-14)

In [ ]:
import os
import sys

# sys.setdlopenflags(RTLD_GLOBAL) MUST run before the first `import duckdb` — Python's
# import machinery otherwise loads DuckDB's native module RTLD_LOCAL, which hides its
# symbols from the ailake.duckdb_extension .so this notebook LOADs in section 10
# (fails with a misleading "undefined symbol" IO error). See duckdb-ailake/README.md
# "Load in Python". Harmless for sections 1-9, which only need standard Parquet scan.
_old_dlopen_flags = sys.getdlopenflags()
sys.setdlopenflags(_old_dlopen_flags | os.RTLD_GLOBAL)
import duckdb
sys.setdlopenflags(_old_dlopen_flags)

import pathlib
import pandas as pd

TABLE_PATH = os.environ.get('DEMO_TABLE_PATH', '/data/ailake_demo')
PARQUET_GLOB = str(pathlib.Path(TABLE_PATH) / '**' / '*.parquet')

con = duckdb.connect(config={"allow_unsigned_extensions": True})
print(f'DuckDB {duckdb.__version__}')
print(f'Parquet glob: {PARQUET_GLOB}')

## 1. Basic scan

In [ ]:
total = con.execute(f"""
    SELECT count(*) AS rows
    FROM read_parquet('{PARQUET_GLOB}', hive_partitioning=false)
""").fetchone()[0]
print(f'Total rows: {total}')

In [ ]:
# Schema: DuckDB exposes the embedding column as BLOB (raw F16 bytes)
con.execute(f"""
    DESCRIBE SELECT * FROM read_parquet('{PARQUET_GLOB}', hive_partitioning=false) LIMIT 1
""").df()

In [ ]:
con.execute(f"""
    SELECT text, octet_length(embedding) AS embedding_bytes
    FROM read_parquet('{PARQUET_GLOB}', hive_partitioning=false)
    LIMIT 5
""").df()

## 2. Filtered queries

In [ ]:
# Full-text keyword filter (pushed down to Parquet column scan)
df = con.execute(f"""
    SELECT text
    FROM read_parquet('{PARQUET_GLOB}', hive_partitioning=false)
    WHERE text LIKE '%vector search%'
""").df()
print(f"Rows mentioning 'vector search': {len(df)}")
df.head(3)

## 3. Aggregations

In [ ]:
con.execute(f"""
    SELECT
        count(*)                    AS total_rows,
        avg(length(text))           AS avg_text_len,
        min(length(text))           AS min_text_len,
        max(length(text))           AS max_text_len,
        avg(octet_length(embedding))      AS avg_emb_bytes,
        sum(octet_length(embedding))      AS total_emb_bytes
    FROM read_parquet('{PARQUET_GLOB}', hive_partitioning=false)
""").df()

## 4. Topic distribution

In [ ]:
# Extract topic keyword (text was generated as "An intro to <topic>: ...")
topics = [
    'machine learning', 'database systems', 'vector search', 'data lakes',
    'Apache Iceberg', 'embedding models', 'RAG pipelines', 'LLM inference',
    'semantic search', 'quantization',
]
rows = []
for t in topics:
    n = con.execute(f"""
        SELECT count(*) FROM read_parquet('{PARQUET_GLOB}', hive_partitioning=false)
        WHERE text LIKE '%{t}%'
    """).fetchone()[0]
    rows.append({'topic': t, 'count': n})
pd.DataFrame(rows).sort_values('count', ascending=False)

## 5. Iceberg extension (optional)

DuckDB's Iceberg extension reads the Iceberg snapshot metadata and discovers
the Parquet files from the manifest — the same as a standard Iceberg catalog read.

In [ ]:
import json

# Resolve latest snapshot metadata path
meta_dir = pathlib.Path(TABLE_PATH) / 'default' / 'table' / 'metadata'
hint = (meta_dir / 'version-hint.text').read_text().strip()
meta_path = meta_dir / f'v{hint}.metadata.json'
meta = json.loads(meta_path.read_text())
print(f'Table UUID    : {meta["table-uuid"]}')
print(f'Format version: {meta["format-version"]}')
print(f'Snapshots     : {len(meta.get("snapshots", []))}')
print()

try:
    con.execute('INSTALL iceberg; LOAD iceberg;')
    iceberg_tbl = str(pathlib.Path(TABLE_PATH) / 'default' / 'table')
    n = con.execute(f"SELECT count(*) FROM iceberg_scan('{iceberg_tbl}')").fetchone()[0]
    print(f'Iceberg extension scan: {n} rows')
except Exception as exc:
    print(f'Iceberg extension unavailable ({exc}) — using Parquet glob scan instead')

## 6. Per-file storage breakdown

Each AI-Lake `.parquet` is self-contained: Parquet data + HNSW footer.
DuckDB reports file sizes; the last ~`hnsw_size` bytes are the index.


In [ ]:
import os

parquet_files = sorted(pathlib.Path(TABLE_PATH).rglob('*.parquet'))
rows = []
for p in parquet_files:
    size = p.stat().st_size
    count = con.execute(
        f"SELECT count(*) FROM read_parquet('{p}', hive_partitioning=false)"
    ).fetchone()[0]
    rows.append({'file': p.name, 'size_bytes': size, 'rows': count,
                 'bytes_per_row': size // max(count, 1)})

import pandas as pd
pd.DataFrame(rows)


## 7. Decode F16 embedding from DuckDB BLOB

The `embedding` column is stored as `FIXED_LEN_BYTE_ARRAY` (F16).
DuckDB exposes it as `BLOB`. Decode back to float32 in Python with `numpy`.


In [ ]:
import numpy as np

# Fetch the raw embedding bytes for row 0
row = con.execute(
    f"SELECT text, embedding FROM read_parquet('{PARQUET_GLOB}', hive_partitioning=false) LIMIT 1"
).fetchone()
text_val, emb_blob = row

# Decode F16 bytes → float32
emb_f16 = np.frombuffer(bytes(emb_blob), dtype=np.float16)
emb_f32 = emb_f16.astype(np.float32)

print(f'Text            : {text_val[:60]}...')
print(f'Embedding bytes : {len(emb_blob)} (= dim × 2 for F16)')
print(f'Decoded dim     : {len(emb_f32)}')
print(f'First 8 values  : {emb_f32[:8].round(4)}')


## 8. Iceberg metadata — snapshot and schema via JSON

Read the Iceberg `metadata.json` directly to inspect table properties
and AI-Lake custom metadata without any Iceberg library.


In [ ]:
import json

meta_dir  = pathlib.Path(TABLE_PATH) / 'default' / 'table' / 'metadata'
hint      = (meta_dir / 'version-hint.text').read_text().strip()
meta_path = meta_dir / f'v{hint}.metadata.json'
meta      = json.loads(meta_path.read_text())

print(f'Table UUID      : {meta["table-uuid"]}')
print(f'Format version  : {meta["format-version"]}')
print(f'Snapshots       : {len(meta.get("snapshots", []))}')
print()
print('AI-Lake properties:')
for k, v in sorted(meta.get('properties', {}).items()):
    if k.startswith('ailake.'):
        print(f'  {k:40s} = {v}')

# Highlight embedding model tracking — present when table was written with embedding_model=
model_key = 'ailake.embedding-model'
if model_key in meta.get('properties', {}):
    print(f'  → Embedding model: {meta["properties"][model_key]}')
else:
    print('  (ailake.embedding-model not set on this table)')


## 9. Embedding model metadata in DuckDB context

When a table is written with `embedding_model=`, the model name is stored in
Iceberg table properties (`ailake.embedding-model`) and in every file's
`key_metadata` in the Avro manifest.

DuckDB reads these as standard Iceberg / Parquet — the model metadata is
accessible directly from the `metadata.json` file or via the Iceberg extension.

In [ ]:
# Show all ailake.* properties (includes embedding model if set)
print('All ailake.* properties for this table:')
for k, v in sorted(meta.get('properties', {}).items()):
    if k.startswith('ailake.'):
        print(f'  {k:45s} = {v}')


## 10. AI-Lake DuckDB Extension — native vector SQL

Everything above reads AI-Lake tables as **plain Parquet** — no plugin, but no vector
search either. The `duckdb-ailake` C++ extension bridges DuckDB to the same C-ABI the
Spark/Trino/Flink plugins use (`libailake_jni.so`), adding `ailake_search()`,
`ailake_scan()`, `ailake_search_multimodal()`, `ailake_search_text()`,
`ailake_write_batch()`, `ailake_write_batch_multi()`, `ailake_delete_where()`,
`ailake_evolve_schema()`, `ailake_compact()` as real SQL table/scalar functions.

Built into this image at `/usr/local/lib/ailake.duckdb_extension` +
`/usr/local/lib/libailake_jni.so` (`Dockerfile`, `-DDUCKDB_VERSION=v1.5.0` matching the
pinned `duckdb==1.5.0` pip package — extension and Python client must match).

In [ ]:
AILAKE_JNI_LIB = os.environ.get('AILAKE_JNI_LIB', '/usr/local/lib/libailake_jni.so')
AILAKE_DUCKDB_EXT = os.environ.get('AILAKE_DUCKDB_EXT', '/usr/local/lib/ailake.duckdb_extension')

import ctypes

if not (pathlib.Path(AILAKE_JNI_LIB).exists() and pathlib.Path(AILAKE_DUCKDB_EXT).exists()):
    print(f'SKIP — extension not built into this image (looked for {AILAKE_DUCKDB_EXT})')
    AILAKE_EXT_LOADED = False
else:
    ctypes.CDLL(AILAKE_JNI_LIB, ctypes.RTLD_GLOBAL)
    con.execute(f"LOAD '{AILAKE_DUCKDB_EXT}'")
    AILAKE_EXT_LOADED = True
    print(f'ailake.duckdb_extension loaded — libailake_jni.so at {AILAKE_JNI_LIB}')

## 11. `ailake_search` + `ailake_scan` — native vector search over SQL

`ailake_search()` returns pointers `(row_id, distance, file_path)` — the same shape as
`01_ailake_demo.ipynb`'s pointer-only `ailake.search()`. `ailake_scan()` does search +
full-row fetch in one native call (backed by `ailake_scan_json`, Fase 11) — no manual
`JOIN` against `parquet_scan()` needed to get `text`/other columns back.

In [ ]:
import json

if not AILAKE_EXT_LOADED:
    print('SKIP')
else:
    QUERY_JSON_PATH = pathlib.Path(TABLE_PATH).parent / 'demo_query.json'
    with open(QUERY_JSON_PATH) as f:
        demo_q = json.load(f)
    query_vec = demo_q['query_vector']

    print('--- ailake_search (pointer-only) ---')
    rows = con.execute(f"""
        SELECT row_id, distance
        FROM ailake_search('file://{TABLE_PATH}', {query_vec}::FLOAT[], 5)
        ORDER BY distance
    """).fetchall()
    for r in rows:
        print(f'  row_id={r[0]:3d}  distance={r[1]:.4f}')

    print()
    print('--- ailake_scan (search + full row, no JOIN) ---')
    df_scan = con.execute(f"""
        SELECT text, _distance
        FROM ailake_scan('file://{TABLE_PATH}', {query_vec}::FLOAT[], 5)
        ORDER BY _distance
    """).df()
    print(df_scan.to_string(index=False))

## 12. `ailake_search_text` — Tantivy FTS over SQL

Pure BM25/Tantivy keyword search (Phase T fast path when the table has a per-file FTS
section, O(log N); falls back to O(N) BM25 scan otherwise) — no embedding required.
Uses the `fts` fixture table (`init_demo.py`, `fts_text_columns=["text"]`).

In [ ]:
if not AILAKE_EXT_LOADED:
    print('SKIP')
else:
    fts_path = demo_q['table_paths']['fts']
    rows = con.execute(f"""
        SELECT row_id, distance
        FROM ailake_search_text('file://{fts_path}', 'machine learning', 5)
        ORDER BY distance
    """).fetchall()
    for r in rows:
        print(f'  row_id={r[0]:3d}  distance={r[1]:.4f}')

## 13. `ailake_search_multimodal` — cross-modal RRF over SQL

Fuses independent HNSW searches across N vector columns via Reciprocal Rank Fusion —
same algorithm as `ailake.search_multimodal()` in `01_ailake_demo.ipynb` §22, from SQL.
Uses the `multimodal` fixture table (`embedding` dim=32 + `image_embedding` dim=16).

In [ ]:
if not AILAKE_EXT_LOADED:
    print('SKIP')
else:
    mm_path = demo_q['table_paths']['multimodal']
    rows = con.execute(f"""
        SELECT row_id, rrf_score
        FROM ailake_search_multimodal(
            'file://{mm_path}',
            [{{'col': 'embedding', 'query': {query_vec}::FLOAT[], 'weight': 1.0}}],
            5
        )
        ORDER BY rrf_score DESC
    """).fetchall()
    for r in rows:
        print(f'  row_id={r[0]:3d}  rrf_score={r[1]:.4f}')

## 14. Write lifecycle from SQL — `ailake_create_table`, `ailake_write_batch`, `ailake_delete_where`, `ailake_evolve_schema`, `ailake_compact`

The extension is not read-only — `ailake_create_table()` provisions an empty schema-only
table, `ailake_write_batch()` ingests new rows, and the same schema-evolution/delete/
compaction primitives from `01_ailake_demo.ipynb` §29-31 are available as scalar
functions returning a status code instead of a Python object.


In [ ]:
if not AILAKE_EXT_LOADED:
    print('SKIP')
else:
    import shutil
    CREATE_PATH = str(pathlib.Path(TABLE_PATH).parent / 'nb_duckdb_create')
    shutil.rmtree(CREATE_PATH, ignore_errors=True)

    created = con.execute(f"SELECT ailake_create_table('file://{CREATE_PATH}', 3, 'embedding', 'cosine')").fetchone()[0]
    print(f'ailake_create_table: {created}')

    # The table is immediately usable by ailake_write_batch — same as the Python SDK's
    # ailake.create_table() followed by open_table() in 01_ailake_demo.ipynb §1B
    snap = con.execute(f"""
        SELECT ailake_write_batch(
            'file://{CREATE_PATH}',
            [0]::BIGINT[],
            [[0.1, 0.2, 0.3]]::FLOAT[][]
        )
    """).fetchone()[0]
    print(f'first write into created table: snapshot_id={snap}')

In [ ]:
if not AILAKE_EXT_LOADED:
    print('SKIP')
else:
    import shutil
    WRITE_PATH = str(pathlib.Path(TABLE_PATH).parent / 'nb_duckdb_write')
    shutil.rmtree(WRITE_PATH, ignore_errors=True)

    snap = con.execute(f"""
        SELECT ailake_write_batch(
            'file://{WRITE_PATH}',
            [0, 1, 2]::BIGINT[],
            [[0.1, 0.2, 0.3], [0.4, 0.5, 0.6], [0.7, 0.8, 0.9]]::FLOAT[][]
        )
    """).fetchone()[0]
    print(f'ailake_write_batch: snapshot_id={snap}')

    rows = con.execute(f"""
        SELECT row_id, distance
        FROM ailake_search('file://{WRITE_PATH}', [0.1, 0.2, 0.3]::FLOAT[], 3)
        ORDER BY distance
    """).fetchall()
    print(f'read-back: {rows}')

    deleted_ok = con.execute(f"SELECT ailake_delete_where('file://{WRITE_PATH}', 'id', ['0'])").fetchone()[0]
    print(f'ailake_delete_where(id=0): {deleted_ok}')

    new_schema_id = con.execute(f"""
        SELECT ailake_evolve_schema(
            'file://{WRITE_PATH}',
            '[{{"name":"note","type":"string","initial_default":null}}]',
            '[]'
        )
    """).fetchone()[0]
    print(f'ailake_evolve_schema (add note): new_schema_id={new_schema_id}')

    files_compacted = con.execute(f"SELECT ailake_compact('file://{WRITE_PATH}', 1)").fetchone()[0]
    print(f'ailake_compact: files_compacted={files_compacted}')

## Key takeaway

DuckDB reads AI-Lake tables as **standard Parquet** — zero configuration, zero plugins.
The HNSW index lives in the file footer after the final `PAR1` magic bytes;
DuckDB stops at `PAR1` and never touches the index section.

The `embedding` column has `octet_length(embedding) = dim * 2` bytes (F16 storage, 2 bytes/dimension).
With dim=32 the demo fixture stores 64 bytes per vector.

For vector similarity search **without leaving SQL**, load the `duckdb-ailake` extension
(§10) — `ailake_search`/`ailake_scan`/`ailake_search_multimodal`/`ailake_search_text`
run the same HNSW/Tantivy/RRF engine as `ailake.search()` (`01_ailake_demo.ipynb`), and
`ailake_create_table`/`ailake_write_batch`/`ailake_delete_where`/`ailake_evolve_schema`/
`ailake_compact` cover the write side too — no Python SDK required.